In [1]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
import os

In [2]:

index_df = pd.read_csv(
    r"D:\DACN\dbl_alphapepdeep\_lt\251128\_index.tsv",
    sep="\t",
    header=None
)

index_df.head()


,0
0,1/17705.tsv
1,1/27289.tsv
2,1/26084.tsv
3,1/7517.tsv
4,1/26511.tsv


In [3]:
print(len(index_df))

484


In [4]:
base_dir = r"D:\DACN\dbl_alphapepdeep\_lt\251128\251128"

dfs = []

for i, rel_path in enumerate(index_df[0]):
    full_path = os.path.join(base_dir, rel_path)
    
    temp = pd.read_csv(full_path, sep="\t")
    temp["psm_id"] = f"psm_{i}"   # 🔥 mỗi file = 1 spectrum
    
    dfs.append(temp)



In [5]:
df = pd.concat(dfs, ignore_index=True)

print(df.head())
print(f"Total rows: {len(df)}")

   PrecursorCharge            ModifiedPeptide FragmentLossType  \
0                2  _AEEDEILNRS(UniMod:21)PR_           noloss   
1                2  _AEEDEILNRS(UniMod:21)PR_           noloss   
2                2  _AEEDEILNRS(UniMod:21)PR_           noloss   
3                2  _AEEDEILNRS(UniMod:21)PR_           noloss   
4                2  _AEEDEILNRS(UniMod:21)PR_           noloss   

   FragmentNumber FragmentType FragmentCharge  RelativeIntensity psm_id  
0               1            y              1           0.085714  psm_0  
1               2            y              1           1.000000  psm_0  
2               3            y              1           0.028571  psm_0  
3               4            y              1           0.014286  psm_0  
4               5            y              1           0.057143  psm_0  
Total rows: 35702


In [6]:
df["FragmentLossType"].unique()


array(['noloss', '1(+H2+O)', '1(+H3+N)', 'H3PO4'], dtype=object)

In [7]:
# df = df[df["FragmentLossType"] == "noloss"].copy()
# df.head()

In [8]:
import re

def convert_modified_peptide(mod_pep):
    mod_pep = mod_pep.strip('_')

    sequence = ""
    mods = []
    i = 0
    pos = 0

    while i < len(mod_pep):
        if mod_pep[i] == '(':
            j = mod_pep.find(')', i)
            mod = mod_pep[i+1:j]

            if "UniMod:21" in mod:
                mods.append(f"Phospho@{sequence[-1]}{pos-1}")

            i = j + 1
        else:
            sequence += mod_pep[i]
            pos += 1
            i += 1

    return sequence, ";".join(mods)

In [9]:
converted = df["ModifiedPeptide"].apply(convert_modified_peptide)
df["sequence"] = converted.apply(lambda x: x[0])
df["mods"] = converted.apply(lambda x: x[1])
df.head()

,PrecursorCharge,ModifiedPeptide,FragmentLossType,FragmentNumber,FragmentType,FragmentCharge,RelativeIntensity,psm_id,sequence,mods
0,2,_AEEDEILNRS(UniMod:21)PR_,noloss,1,y,1,0.085714,psm_0,AEEDEILNRSPR,Phospho@S9
1,2,_AEEDEILNRS(UniMod:21)PR_,noloss,2,y,1,1.000000,psm_0,AEEDEILNRSPR,Phospho@S9
2,2,_AEEDEILNRS(UniMod:21)PR_,noloss,3,y,1,0.028571,psm_0,AEEDEILNRSPR,Phospho@S9
3,2,_AEEDEILNRS(UniMod:21)PR_,noloss,4,y,1,0.014286,psm_0,AEEDEILNRSPR,Phospho@S9
4,2,_AEEDEILNRS(UniMod:21)PR_,noloss,5,y,1,0.057143,psm_0,AEEDEILNRSPR,Phospho@S9


In [10]:
def build_frag_col(row):
    ion = row["FragmentType"]     # b hoặc y
    charge = int(
        str(row["FragmentCharge"])
            .replace("(", "")
            .replace(")", "")
            .replace("+", "")
    )
    loss = row["FragmentLossType"]

    if loss == "noloss":
        return f"{ion}_z{charge}"
    else:
        return f"{ion}_modloss_z{charge}"

df["frag_col"] = df.apply(build_frag_col, axis=1)

In [11]:
df.head()

,PrecursorCharge,ModifiedPeptide,FragmentLossType,FragmentNumber,FragmentType,FragmentCharge,RelativeIntensity,psm_id,sequence,mods,frag_col
0,2,_AEEDEILNRS(UniMod:21)PR_,noloss,1,y,1,0.085714,psm_0,AEEDEILNRSPR,Phospho@S9,y_z1
1,2,_AEEDEILNRS(UniMod:21)PR_,noloss,2,y,1,1.000000,psm_0,AEEDEILNRSPR,Phospho@S9,y_z1
2,2,_AEEDEILNRS(UniMod:21)PR_,noloss,3,y,1,0.028571,psm_0,AEEDEILNRSPR,Phospho@S9,y_z1
3,2,_AEEDEILNRS(UniMod:21)PR_,noloss,4,y,1,0.014286,psm_0,AEEDEILNRSPR,Phospho@S9,y_z1
4,2,_AEEDEILNRS(UniMod:21)PR_,noloss,5,y,1,0.057143,psm_0,AEEDEILNRSPR,Phospho@S9,y_z1


In [12]:
df.groupby("psm_id")["ModifiedPeptide"].nunique().value_counts()

ModifiedPeptide
1    484
Name: count, dtype: int64

In [13]:
df["frag_col"].value_counts()

frag_col
y_modloss_z1    11360
b_modloss_z1    11280
y_z1             7255
b_z1             5279
y_z2              286
b_z2              242
Name: count, dtype: int64

In [14]:
pivot_df = df.pivot_table(
    index=["psm_id", "FragmentNumber"],
    columns="frag_col",
    values="RelativeIntensity",
    aggfunc="max", 
    fill_value=0
).reset_index()
pivot_df.head()

frag_col,psm_id,FragmentNumber,b_modloss_z1,b_z1,b_z2,y_modloss_z1,y_z1,y_z2
0,psm_0,1,0.000000,0.000000,0.0,0.005714,0.085714,0.0
1,psm_0,2,0.000000,0.085714,0.0,0.014286,1.000000,0.0
2,psm_0,3,0.004286,0.014286,0.0,0.028571,0.028571,0.0
3,psm_0,4,0.014286,0.042857,0.0,0.012857,0.014286,0.0
4,psm_0,5,0.014286,0.071429,0.0,0.042857,0.057143,0.0


In [15]:
meta = df.groupby("psm_id").agg({
    "sequence": "first",
    "mods": "first",
    "PrecursorCharge": "first"
}).reset_index()

pivot_df = pivot_df.merge(meta, on="psm_id", how="left")

pivot_df.head() 

,psm_id,FragmentNumber,b_modloss_z1,b_z1,b_z2,y_modloss_z1,y_z1,y_z2,sequence,mods,PrecursorCharge
0,psm_0,1,0.000000,0.000000,0.0,0.005714,0.085714,0.0,AEEDEILNRSPR,Phospho@S9,2
1,psm_0,2,0.000000,0.085714,0.0,0.014286,1.000000,0.0,AEEDEILNRSPR,Phospho@S9,2
2,psm_0,3,0.004286,0.014286,0.0,0.028571,0.028571,0.0,AEEDEILNRSPR,Phospho@S9,2
3,psm_0,4,0.014286,0.042857,0.0,0.012857,0.014286,0.0,AEEDEILNRSPR,Phospho@S9,2
4,psm_0,5,0.014286,0.071429,0.0,0.042857,0.057143,0.0,AEEDEILNRSPR,Phospho@S9,2


In [16]:

pivot_df.columns = pivot_df.columns.str.replace("(", "", regex=False)
pivot_df.columns = pivot_df.columns.str.replace(")", "", regex=False)
pivot_df.columns = pivot_df.columns.str.replace("+", "", regex=False)

pivot_df["b_modloss_z2"] = 0.0
pivot_df["y_modloss_z2"] = 0.0

pivot_df.head(10)

,psm_id,FragmentNumber,b_modloss_z1,b_z1,b_z2,y_modloss_z1,y_z1,y_z2,sequence,mods,PrecursorCharge,b_modloss_z2,y_modloss_z2
0,psm_0,1,0.000000,0.000000,0.0,0.005714,0.085714,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
1,psm_0,2,0.000000,0.085714,0.0,0.014286,1.000000,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
2,psm_0,3,0.004286,0.014286,0.0,0.028571,0.028571,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
3,psm_0,4,0.014286,0.042857,0.0,0.012857,0.014286,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
4,psm_0,5,0.014286,0.071429,0.0,0.042857,0.057143,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
5,psm_0,6,0.011429,0.042857,0.0,0.042857,0.085714,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
6,psm_0,7,0.002857,0.012857,0.0,0.071429,0.057143,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
7,psm_0,8,0.002857,0.005714,0.0,0.285714,0.114286,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
8,psm_0,9,0.011429,0.057143,0.0,0.100000,0.057143,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
9,psm_0,10,1.000000,0.142857,0.0,0.142857,0.085714,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0


In [17]:
print(df["psm_id"].nunique())
print(pivot_df["psm_id"].nunique())

484
484


In [18]:
pivot_df.groupby("psm_id")["sequence"].nunique().value_counts()

sequence
1    484
Name: count, dtype: int64

In [19]:
pivot_df.groupby(["psm_id", "FragmentNumber"]).size().value_counts()

1    8025
Name: count, dtype: int64

In [20]:
def ensure_full_fragment(group):

    seq = group["sequence"].iloc[0]
    mods = group["mods"].iloc[0]
    charge = group["PrecursorCharge"].iloc[0]
    psm_id = group["psm_id"].iloc[0]

    n_frag = len(seq) - 1

    full_index = pd.DataFrame({
        "FragmentNumber": range(1, n_frag + 1)
    })

    group = full_index.merge(group, on="FragmentNumber", how="left")

    intensity_cols = ["b_z1", "y_z1",]

    group[intensity_cols] = group[intensity_cols].fillna(0)

    group["psm_id"] = psm_id
    group["sequence"] = seq
    group["mods"] = mods
    group["PrecursorCharge"] = charge

    return group

pivot_df = pivot_df.groupby("psm_id").apply(ensure_full_fragment).reset_index(drop=True)

C:\Users\letie\AppData\Local\Temp\ipykernel_16228\2503600903.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pivot_df = pivot_df.groupby("psm_id").apply(ensure_full_fragment).reset_index(drop=True)


In [21]:
pivot_df.head()

,FragmentNumber,psm_id,b_modloss_z1,b_z1,b_z2,y_modloss_z1,y_z1,y_z2,sequence,mods,PrecursorCharge,b_modloss_z2,y_modloss_z2
0,1,psm_0,0.000000,0.000000,0.0,0.005714,0.085714,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
1,2,psm_0,0.000000,0.085714,0.0,0.014286,1.000000,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
2,3,psm_0,0.004286,0.014286,0.0,0.028571,0.028571,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
3,4,psm_0,0.014286,0.042857,0.0,0.012857,0.014286,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
4,5,psm_0,0.014286,0.071429,0.0,0.042857,0.057143,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0


In [22]:
check = pivot_df.groupby("psm_id").apply(
    lambda x: x["FragmentNumber"].nunique() == (len(x["sequence"].iloc[0]) - 1)
)

print(check.value_counts())

True    484
Name: count, dtype: int64


C:\Users\letie\AppData\Local\Temp\ipykernel_16228\2348494221.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  check = pivot_df.groupby("psm_id").apply(


In [23]:
pivot_df = pivot_df.sort_values(["psm_id", "FragmentNumber"])
pivot_df.head()

,FragmentNumber,psm_id,b_modloss_z1,b_z1,b_z2,y_modloss_z1,y_z1,y_z2,sequence,mods,PrecursorCharge,b_modloss_z2,y_modloss_z2
0,1,psm_0,0.000000,0.000000,0.0,0.005714,0.085714,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
1,2,psm_0,0.000000,0.085714,0.0,0.014286,1.000000,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
2,3,psm_0,0.004286,0.014286,0.0,0.028571,0.028571,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
3,4,psm_0,0.014286,0.042857,0.0,0.012857,0.014286,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0
4,5,psm_0,0.014286,0.071429,0.0,0.042857,0.057143,0.0,AEEDEILNRSPR,Phospho@S9,2,0.0,0.0


In [24]:
intensity_cols = ["b_z1","y_z1","b_modloss_z1", "y_modloss_z1", "b_z2", "y_z2", "b_modloss_z2", "y_modloss_z2"]

psm_rows = []
fragment_rows = []

current_idx = 0

for psm_id, group in pivot_df.groupby("psm_id"):

    sequence = group["sequence"].iloc[0]
    mods = group["mods"].iloc[0]
    charge = group["PrecursorCharge"].iloc[0]

    n_frag = len(sequence) - 1

    # Build psm_df row
    psm_rows.append({
        "sequence": sequence,
        "mods": mods,
        "charge": charge,
        "nAA": len(sequence),
        "frag_start_idx": current_idx,
        "frag_stop_idx": current_idx + n_frag,
        "nce": 30.0,
        "instrument": "Lumos"
    })

    # Append fragment intensities
    fragment_rows.append(group[intensity_cols].iloc[:n_frag])

    current_idx += n_frag

psm_df = pd.DataFrame(psm_rows)
matched_intensity_df = pd.concat(fragment_rows, ignore_index=True)

In [25]:
print(len(psm_df))
print(len(matched_intensity_df))
print(sum(psm_df["nAA"] - 1))
print(matched_intensity_df.sum().sum())

484
8356
8356
3461.9553657394003


In [26]:
matched_intensity_df.head()

,b_z1,y_z1,b_modloss_z1,y_modloss_z1,b_z2,y_z2,b_modloss_z2,y_modloss_z2
0,0.000000,0.085714,0.000000,0.005714,0.0,0.0,0.0,0.0
1,0.085714,1.000000,0.000000,0.014286,0.0,0.0,0.0,0.0
2,0.014286,0.028571,0.004286,0.028571,0.0,0.0,0.0,0.0
3,0.042857,0.014286,0.014286,0.012857,0.0,0.0,0.0,0.0
4,0.071429,0.057143,0.014286,0.042857,0.0,0.0,0.0,0.0


In [27]:
len(psm_df), len(matched_intensity_df)

(484, 8356)

In [28]:
print("Number of PSM:", len(psm_df))
print("Unique peptides:", psm_df["sequence"].nunique())
print("Matched intensity shape:", matched_intensity_df.shape)

assert (psm_df["frag_stop_idx"] - psm_df["frag_start_idx"]).equals(psm_df["nAA"] - 1)

Number of PSM: 484
Unique peptides: 50
Matched intensity shape: (8356, 8)


In [29]:
print(psm_df["mods"].unique())

['Phospho@S9' 'Phospho@S7' 'Phospho@S10' 'Phospho@S1' 'Phospho@S4'
 'Phospho@S2' 'Phospho@S6' 'Phospho@T2' 'Phospho@S21' 'Phospho@S15'
 'Phospho@S8' 'Phospho@Y4;Phospho@T8;Phospho@T10' 'Phospho@T4'
 'Phospho@S5' 'Phospho@S3' 'Phospho@S12' 'Phospho@S4;Phospho@S13'
 'Phospho@S4;Phospho@S6' 'Phospho@T8' 'Phospho@T3' 'Phospho@S16']


In [30]:
def split_mod_and_site(mod_string):
    if not mod_string or mod_string == 0:
        return "", ""
    
    mod_list = []
    site_list = []
    
    for mod in mod_string.split(";"):
        if "Phospho@" in mod:
            aa = mod.split("@")[1][0]      # S
            pos = mod.split("@")[1][1:]    # 21
            
            mod_list.append(f"Phospho@{aa}")
            site_list.append(pos)
    
    return ";".join(mod_list), ";".join(site_list)


psm_df[["mods", "mod_sites"]] = (
    psm_df["mods"]
    .apply(lambda x: pd.Series(split_mod_and_site(x)))
)

In [31]:
print(psm_df["mods"].unique())
print(psm_df["mod_sites"].unique())

['Phospho@S' 'Phospho@T' 'Phospho@Y;Phospho@T;Phospho@T'
 'Phospho@S;Phospho@S']
['9' '7' '10' '1' '4' '2' '6' '21' '15' '8' '4;8;10' '5' '3' '12' '4;13'
 '4;6' '16']


In [32]:
psm_df.head()

,sequence,mods,charge,nAA,frag_start_idx,frag_stop_idx,nce,instrument,mod_sites
0,AEEDEILNRSPR,Phospho@S,2,12,0,11,30.0,Lumos,9
1,AEEDEILNRSPR,Phospho@S,2,12,11,22,30.0,Lumos,9
2,AGDLLEDSPKRPK,Phospho@S,2,13,22,34,30.0,Lumos,7
3,ASGQAFELILSPR,Phospho@S,2,13,34,46,30.0,Lumos,10
4,ASGQAFELILSPR,Phospho@S,2,13,46,58,30.0,Lumos,10


In [33]:
matched_intensity_df.describe()

,b_z1,y_z1,b_modloss_z1,y_modloss_z1,b_z2,y_z2,b_modloss_z2,y_modloss_z2
count,8356.000000,8356.000000,8025.000000,8025.000000,8025.000000,8025.000000,8025.0,8025.0
mean,0.064332,0.172873,0.067172,0.115771,0.000465,0.001000,0.0,0.0
std,0.143195,0.238299,0.117789,0.217387,0.004730,0.008202,0.0,0.0
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
25%,0.000000,0.020000,0.002000,0.000000,0.000000,0.000000,0.0,0.0
50%,0.011111,0.080000,0.022500,0.021020,0.000000,0.000000,0.0,0.0
75%,0.050000,0.209297,0.080000,0.100000,0.000000,0.000000,0.0,0.0
max,1.000000,1.000000,1.000000,1.000000,0.187603,0.200000,0.0,0.0


In [34]:
(matched_intensity_df == 0).mean()

b_z1            0.368238
y_z1            0.131762
b_modloss_z1    0.230972
y_modloss_z1    0.252753
b_z2            0.931427
y_z2            0.926161
b_modloss_z2    0.960388
y_modloss_z2    0.960388
dtype: float64

In [35]:
# Check
zero_percent = (matched_intensity_df.values == 0).sum() / matched_intensity_df.values.size * 100
print(f"Zero percentage: {zero_percent:.1f}%")

# Nếu > 80% zeros → Data rất sparse, khó train
# Solution: Thêm small noise hoặc filter PSMs có ít zeros

Zero percentage: 59.5%


In [36]:
assert psm_df['frag_stop_idx'].max() == len(matched_intensity_df), \
    f"Total fragments mismatch: {psm_df['frag_stop_idx'].max()} != {len(matched_intensity_df)}"

In [37]:
import pandas as pd
import numpy as np

def validate_ms2_dataframes(psm_df, matched_intensity_df):
    """Kiểm tra xem 2 dataframe có hợp lệ không"""
    
    print("=" * 60)
    print("VALIDATING MS2 DATAFRAMES")
    print("=" * 60)
    
    # 1. Check required columns in psm_df
    required_cols = ['sequence', 'mods', 'mod_sites', 'charge', 
                     'nce', 'instrument', 'nAA', 'frag_start_idx', 'frag_stop_idx']
    missing_cols = [col for col in required_cols if col not in psm_df.columns]
    if missing_cols:
        print(f"❌ Missing columns in psm_df: {missing_cols}")
        return False
    print("✓ All required columns in psm_df")
    
    # 2. Check shapes
    print(f"\n📊 Shapes:")
    print(f"   psm_df: {psm_df.shape}")
    print(f"   matched_intensity_df: {matched_intensity_df.shape}")
    
    # 3. Check fragment indices
    print(f"\n🔗 Fragment indices:")
    print(f"   frag_start_idx min: {psm_df['frag_start_idx'].min()}")
    print(f"   frag_start_idx max: {psm_df['frag_start_idx'].max()}")
    print(f"   frag_stop_idx min: {psm_df['frag_stop_idx'].min()}")
    print(f"   frag_stop_idx max: {psm_df['frag_stop_idx'].max()}")
    print(f"   matched_intensity_df rows: {len(matched_intensity_df)}")
    
    # 4. Main validation
    if psm_df['frag_stop_idx'].max() != len(matched_intensity_df):
        print(f"\n❌ Fragment count mismatch!")
        print(f"   Expected: {psm_df['frag_stop_idx'].max()}")
        print(f"   Got: {len(matched_intensity_df)}")
        return False
    
    print(f"\n✓ Fragment count matches: {len(matched_intensity_df)}")
    
    # 5. Check fragment types
    print(f"\n📋 Fragment types:")
    print(f"   Columns: {list(matched_intensity_df.columns)}")
    
    # 6. Check each peptide
    print(f"\n🔍 Detailed check per peptide:")
    for idx, row in psm_df.iterrows():
        seq = row['sequence']
        nAA = row['nAA']
        expected_frags = nAA - 1
        start = row['frag_start_idx']
        stop = row['frag_stop_idx']
        actual_frags = stop - start
        
        print(f"   Peptide {idx}: {seq}")
        print(f"      nAA: {nAA}, Expected fragments: {expected_frags}, Actual: {actual_frags}", end="")
        
        if expected_frags != actual_frags:
            print(" ❌ MISMATCH")
            return False
        else:
            print(" ✓")
    
    print("\n" + "=" * 60)
    print("✓ ALL VALIDATIONS PASSED!")
    print("=" * 60)
    return True

# Sử dụng:
is_valid = validate_ms2_dataframes(psm_df, matched_intensity_df)

VALIDATING MS2 DATAFRAMES
✓ All required columns in psm_df

📊 Shapes:
   psm_df: (484, 9)
   matched_intensity_df: (8356, 8)

🔗 Fragment indices:
   frag_start_idx min: 0
   frag_start_idx max: 8347
   frag_stop_idx min: 11
   frag_stop_idx max: 8356
   matched_intensity_df rows: 8356

✓ Fragment count matches: 8356

📋 Fragment types:
   Columns: ['b_z1', 'y_z1', 'b_modloss_z1', 'y_modloss_z1', 'b_z2', 'y_z2', 'b_modloss_z2', 'y_modloss_z2']

🔍 Detailed check per peptide:
   Peptide 0: AEEDEILNRSPR
      nAA: 12, Expected fragments: 11, Actual: 11 ✓
   Peptide 1: AEEDEILNRSPR
      nAA: 12, Expected fragments: 11, Actual: 11 ✓
   Peptide 2: AGDLLEDSPKRPK
      nAA: 13, Expected fragments: 12, Actual: 12 ✓
   Peptide 3: ASGQAFELILSPR
      nAA: 13, Expected fragments: 12, Actual: 12 ✓
   Peptide 4: ASGQAFELILSPR
      nAA: 13, Expected fragments: 12, Actual: 12 ✓
   Peptide 5: ASGQAFELILSPR
      nAA: 13, Expected fragments: 12, Actual: 12 ✓
   Peptide 6: ASGQAFELILSPR
      nAA: 13, Ex

In [2]:
from peptdeep.pretrained_models import ModelManager
from peptdeep.model.ms2 import normalize_fragment_intensities
import pandas as pd

print("="*70)
print("FULL RETRAIN - CORRECT TRANSFER LEARNING")
print("="*70)

# STEP 1: Fresh ModelManager
print("\n1. Creating ModelManager...")
model_mgr = ModelManager(device='cpu')
print("✓ Created")
model_mgr.verbose = True

# STEP 2: Load PHOSPHO pretrained model
print("\n2. Loading PHOSPHO model...")
model_mgr.load_installed_models(model_type='phospho')
print("✓ Phospho model loaded")

# STEP 3: Verify fragments
print("\n3. Using pretrained fragment types:")
charged_frag_types = model_mgr.ms2_model.model.supported_charged_frag_types
print(f"✓ Fragment types: {charged_frag_types}")

# STEP 4: Match intensity df
print("\n4. Align matched_intensity_df...")
matched_intensity_df_full = matched_intensity_df[charged_frag_types].copy()
print(f"✓ Shape: {matched_intensity_df_full.shape}")

# # STEP 5: NORMALIZE INTENSITIES
# print("\n5. Normalizing intensities...")
# normalize_fragment_intensities(psm_df, matched_intensity_df_full)
# print("✓ Normalized")

# STEP 6: Configure training
print("\n6. Configuring training parameters...")
model_mgr.psm_num_to_train_ms2 = len(psm_df)
model_mgr.psm_num_to_test_ms2 = 0  # ✅ TRAIN HẾT, KHÔNG TEST
model_mgr.epoch_to_train_ms2 = 20
model_mgr.batch_size_to_train_ms2 = 64
model_mgr.lr_to_train_ms2 = 1e-4
model_mgr.warmup_epoch_to_train_ms2 = 1
model_mgr.use_grid_nce_search = False
model_mgr.nce = 30.0
model_mgr.instrument = 'Lumos'

print(f"✓ Training {len(psm_df)} PSMs")
print(f"  Epochs: {model_mgr.epoch_to_train_ms2}")
print(f"  Batch size: {model_mgr.batch_size_to_train_ms2}")
print(f"  LR: {model_mgr.lr_to_train_ms2}")

# STEP 7: TRAIN
print("\n7. TRAINING...")
print("="*70)

model_mgr.train_ms2_model(psm_df, matched_intensity_df_full)

print("="*70)
print("\n✓ Training completed!")

# STEP 8: Save model
print("\n8. Saving model...")
model_mgr.ms2_model.save("ms2_phospho_retrained.pth")
print("✓ Model saved!")

# STEP 9: CHECK METRICS trên training data
print("\n9. Checking metrics...")
metrics = model_mgr.ms2_model.test(psm_df, matched_intensity_df_full)
print("\n📊 METRICS:")
print(metrics)

2026-02-28 20:21:36> Enabling RDKit 2025.09.1 jupyter extensions
FULL RETRAIN - CORRECT TRANSFER LEARNING

1. Creating ModelManager...


D:\DACN\dbl_alphapepdeep\alphapeptdeep\peptdeep\model\ms2.py:416: UserWarning: mask_modloss is deprecated and will be removed in the future. To mask the modloss fragments, the charged_frag_types should not include the modloss fragments.
  warnings.warn(


✓ Created

2. Loading PHOSPHO model...
✓ Phospho model loaded

3. Using pretrained fragment types:
✓ Fragment types: ['b_z1', 'b_z2', 'y_z1', 'y_z2', 'b_modloss_z1', 'b_modloss_z2', 'y_modloss_z1', 'y_modloss_z2']

4. Align matched_intensity_df...


NameError: name 'matched_intensity_df' is not defined

In [4]:
sum(p.numel() for p in model_mgr.ms2_model.parameters() if p.requires_grad)

AttributeError: 'pDeepModel' object has no attribute 'parameters'

In [39]:
import numpy as np

print("="*70)
print("DEBUG: CHECK INTENSITIES")
print("="*70)

# 1. Check matched_intensity_df trước normalize
print("\n1. Matched intensities BEFORE normalize:")
print(f"   Min: {matched_intensity_df[charged_frag_types].min().min()}")
print(f"   Max: {matched_intensity_df[charged_frag_types].max().max()}")
print(f"   Mean: {matched_intensity_df[charged_frag_types].mean().mean()}")
print(f"   Std: {matched_intensity_df[charged_frag_types].std().mean()}")
print(f"\n   Sample:")
print(matched_intensity_df[charged_frag_types].head())

# 2. Normalize
normalize_fragment_intensities(psm_df, matched_intensity_df_full)

# 3. Check AFTER normalize
print("\n2. Matched intensities AFTER normalize:")
print(f"   Min: {matched_intensity_df_full.min().min()}")
print(f"   Max: {matched_intensity_df_full.max().max()}")
print(f"   Mean: {matched_intensity_df_full.mean().mean()}")
print(f"   Std: {matched_intensity_df_full.std().mean()}")
print(f"   Has NaN: {matched_intensity_df_full.isna().sum().sum()}")
print(f"   All zeros: {(matched_intensity_df_full == 0).sum().sum()}")

# 4. Check per peptide
print("\n3. Check per peptide (first 3):")
for i in range(min(3, len(psm_df))):
    start = psm_df.iloc[i]['frag_start_idx']
    stop = psm_df.iloc[i]['frag_stop_idx']
    seq = psm_df.iloc[i]['sequence']
    frag_data = matched_intensity_df_full.iloc[start:stop]
    print(f"\n   Peptide {i}: {seq}")
    print(f"      Frags {start}-{stop}: min={frag_data.min().min():.6f}, max={frag_data.max().max():.6f}")
    print(f"      Sum per frag: {frag_data.sum(axis=1).values[:3]}")

DEBUG: CHECK INTENSITIES

1. Matched intensities BEFORE normalize:
   Min: 0.0
   Max: 1.0
   Mean: 0.052701566019651314
   Std: 0.09120016668018244

   Sample:
       b_z1  b_z2      y_z1  y_z2  b_modloss_z1  b_modloss_z2  y_modloss_z1  \
0  0.000000   0.0  0.085714   0.0      0.000000           0.0      0.005714   
1  0.085714   0.0  1.000000   0.0      0.000000           0.0      0.014286   
2  0.014286   0.0  0.028571   0.0      0.004286           0.0      0.028571   
3  0.042857   0.0  0.014286   0.0      0.014286           0.0      0.012857   
4  0.071429   0.0  0.057143   0.0      0.014286           0.0      0.042857   

   y_modloss_z2  
0           0.0  
1           0.0  
2           0.0  
3           0.0  
4           0.0  

2. Matched intensities AFTER normalize:
   Min: 0.0
   Max: 1.0
   Mean: 0.052701566019651314
   Std: 0.09120016668018244
   Has NaN: 1986
   All zeros: 39792

3. Check per peptide (first 3):

   Peptide 0: AEEDEILNRSPR
      Frags 0-11: min=0.000000, max

In [40]:
print(model_mgr.ms2_model.model.supported_charged_frag_types)
print(matched_intensity_df_full.columns.tolist())

['b_z1', 'b_z2', 'y_z1', 'y_z2', 'b_modloss_z1', 'b_modloss_z2', 'y_modloss_z1', 'y_modloss_z2']
['b_z1', 'b_z2', 'y_z1', 'y_z2', 'b_modloss_z1', 'b_modloss_z2', 'y_modloss_z1', 'y_modloss_z2']


In [41]:
# Check
zero_percent = (matched_intensity_df.values == 0).sum() / matched_intensity_df.values.size * 100
print(f"Zero percentage: {zero_percent:.1f}%")

# Nếu > 80% zeros → Data rất sparse, khó train
# Solution: Thêm small noise hoặc filter PSMs có ít zeros

Zero percentage: 59.5%


In [42]:
# Kiểm tra xem intensities đã normalize chưa
from peptdeep.model.ms2 import normalize_fragment_intensities

# Normalize again nếu cần
normalize_fragment_intensities(psm_df, matched_intensity_df)
print("✓ Intensities re-normalized")

✓ Intensities re-normalized


In [1]:
import os
import pandas as pd
import numpy as np
from alphabase.spectral_library.reader import LibraryReaderBase
from alphabase.peptide.fragment import concat_precursor_fragment_dataframes
from peptdeep.pretrained_models import ModelManager

print("="*70)
print("MS2 TRAINING - CORRECT METHOD")
print("="*70)

# STEP 1: Load data (như thầy bạn)
print("\n1. Loading data...")
dfs = []
frag_inten_dfs = []

_lib = LibraryReaderBase()
base_path = r"D:\DACN\dbl_alphapepdeep\_lt\251128\251128"

with open(r"D:\DACN\dbl_alphapepdeep\_lt\251128\_index.tsv", "r") as file:
    for line in file:
        file_name = line.rstrip()
        full_path = os.path.join(base_path, file_name)
        a = _lib.import_file(full_path)
        dfs.append(a)
        frag_inten_dfs.append(_lib.fragment_intensity_df)

psm_df, frag_df = concat_precursor_fragment_dataframes(dfs, frag_inten_dfs)
print(f"✓ Loaded: {len(psm_df)} PSMs, {len(frag_df)} fragments")

# STEP 2: Initialize ModelManager
print("\n2. Initializing ModelManager...")
model_mgr = ModelManager(device='cpu')
model_mgr.load_installed_models(model_type='phospho')
print("✓ Model loaded")

# STEP 3: Get supported fragment types
supported_fragment_types = model_mgr.ms2_model.model.supported_charged_frag_types
print(f"✓ Fragment types: {supported_fragment_types}")

# STEP 4: Set NCE and instrument
print("\n3. Setting NCE and instrument...")
psm_df['nAA'] = psm_df['sequence'].str.len()
psm_df['nce'] = 30.0
psm_df['instrument'] = "Lumos"
psm_df['precursor_idx'] = psm_df.index
print("✓ Set")

# STEP 5: ✅ TRAIN (direct method!)
print("\n4. TRAINING...")
print("="*70)

model_mgr.ms2_model.train(
    precursor_df=psm_df,
    fragment_intensity_df=frag_df.loc[:, supported_fragment_types],
    batch_size=64,
    epoch=30,
    verbose=True
)

print("="*70)
print("✓ Training completed!")

# STEP 6: EVALUATE
print("\n5. EVALUATING...")

def calculate_similarity(precursor_df_a, precursor_df_b, intensity_df_a, intensity_df_b):
    """Calculate cosine similarity between predicted and observed"""
    _a_df = precursor_df_a[['precursor_idx', 'frag_start_idx', 'frag_stop_idx']].copy()
    _b_df = precursor_df_b[['precursor_idx', 'frag_start_idx', 'frag_stop_idx']].copy()
    _merged_df = pd.merge(_a_df, _b_df, on='precursor_idx', suffixes=('_a', '_b'))
    _merged_df = _merged_df.drop_duplicates(subset='precursor_idx', keep='first')
    
    similarity_list = []
    for i, (start_a, stop_a, start_b, stop_b) in enumerate(
        zip(_merged_df['frag_start_idx_a'], _merged_df['frag_stop_idx_a'], 
            _merged_df['frag_start_idx_b'], _merged_df['frag_stop_idx_b'])
    ):
        observed = intensity_df_a.loc[start_a:stop_a, :].values.flatten()
        predicted = intensity_df_b.loc[start_b:stop_b, :].values.flatten()
        similarity = np.dot(observed, predicted) / (
            (np.linalg.norm(observed) * np.linalg.norm(predicted)) + 1e-6
        )
        similarity_list.append({
            'similarity': similarity,
            'index': i,
            'precursor_idx': _merged_df.iloc[i]['precursor_idx']
        })
    
    return pd.DataFrame(similarity_list)

# Predict
print("\n6. PREDICTING...")
prediction_precursor_df = psm_df.copy()
prediction_precursor_df.drop(columns=["frag_start_idx", "frag_stop_idx"], inplace=True)  # ✅ DROP!

predictions = model_mgr.ms2_model.predict(prediction_precursor_df)

# Calculate similarity
similarity_df = calculate_similarity(
    psm_df, 
    prediction_precursor_df, 
    frag_df.loc[:, supported_fragment_types], 
    predictions
)

print(f"\n📊 RESULTS:")
print(f"   Median similarity: {similarity_df['similarity'].median():.4f}")
print(f"   Mean similarity: {similarity_df['similarity'].mean():.4f}")
print(f"   Max similarity: {similarity_df['similarity'].max():.4f}")
print(f"   Min similarity: {similarity_df['similarity'].min():.4f}")

# STEP 7: SAVE
print("\n7. SAVING MODEL...")
model_mgr.ms2_model.save("ms2_phospho_retrained.pth")
print("✓ Model saved!")

MS2 TRAINING - CORRECT METHOD

1. Loading data...


c:\Users\letie\anaconda3\envs\peptdeep_env\lib\site-packages\alphabase\psm_reader\psm_reader.py:173: FutureWarning: Passed unknown arguments to LibraryReaderBase (fixed_C57=False) will be forbidden in alphabase>1.5.0.
  warnings.warn(
100%|██████████| 1/1 [00:00<00:00, 286.55it/s]


✓ Loaded: 484 PSMs, 8356 fragments

2. Initializing ModelManager...


D:\DACN\dbl_alphapepdeep\alphapeptdeep\peptdeep\model\ms2.py:416: UserWarning: mask_modloss is deprecated and will be removed in the future. To mask the modloss fragments, the charged_frag_types should not include the modloss fragments.
  warnings.warn(


✓ Model loaded
✓ Fragment types: ['b_z1', 'b_z2', 'y_z1', 'y_z2', 'b_modloss_z1', 'b_modloss_z2', 'y_modloss_z1', 'y_modloss_z2']

3. Setting NCE and instrument...
✓ Set

4. TRAINING...
2026-02-27 15:28:37> Training with fixed sequence length: 0
[Training] Epoch=1, Mean Loss=0.04160253263332627
[Training] Epoch=2, Mean Loss=0.038675118957392195
[Training] Epoch=3, Mean Loss=0.03657563166184859
[Training] Epoch=4, Mean Loss=0.035062453116882934
[Training] Epoch=5, Mean Loss=0.03279904432764107
[Training] Epoch=6, Mean Loss=0.0316623081876473
[Training] Epoch=7, Mean Loss=0.02994013416834853
[Training] Epoch=8, Mean Loss=0.028990040998905897
[Training] Epoch=9, Mean Loss=0.027865199964832176
[Training] Epoch=10, Mean Loss=0.02638842999427156
[Training] Epoch=11, Mean Loss=0.025009007362479515
[Training] Epoch=12, Mean Loss=0.024054726458747278
[Training] Epoch=13, Mean Loss=0.023026475132527677
[Training] Epoch=14, Mean Loss=0.021938142070377416
[Training] Epoch=15, Mean Loss=0.021194472

In [46]:
def calculate_similarity(precursor_df_a, precursor_df_b, intensity_df_a, intensity_df_b):
    """Calculate cosine similarity (CÁCH THẦY)"""
    _a_df = precursor_df_a[['frag_start_idx', 'frag_stop_idx']].copy()
    _b_df = precursor_df_b[['frag_start_idx', 'frag_stop_idx']].copy()
    
    similarity_list = []
    for i in range(len(_a_df)):
        start_a, stop_a = _a_df.iloc[i]
        start_b, stop_b = _b_df.iloc[i]
        
        observed = intensity_df_a.loc[start_a:stop_a, :].values.flatten()
        predicted = intensity_df_b.loc[start_b:stop_b, :].values.flatten()
        
        similarity = np.dot(observed, predicted) / (
            (np.linalg.norm(observed) * np.linalg.norm(predicted)) + 1e-6
        )
        similarity_list.append(similarity)
    
    return pd.Series(similarity_list)

In [49]:
# 1. Train và check loss
print("TRAINING...")
model_mgr.ms2_model.train(
    precursor_df=psm_df,
    fragment_intensity_df=frag_df.loc[:, supported_fragment_types],
    batch_size=64,
    epoch=5,  # ↓ Giảm epoch để xem loss nhanh
    verbose=True
)

# 2. Predict 1 PSM
print("\nPREDICT 1 PSM...")
test_psm = psm_df.iloc[:1].copy()
test_psm.drop(columns=["frag_start_idx", "frag_stop_idx"], inplace=True)
test_pred = model_mgr.ms2_model.predict(test_psm)

print(f"Predicted shape: {test_pred.shape}")
print(f"Predicted values: {test_pred}")
print(f"All zeros: {(test_pred == 0).all().all()}")

# 3. Compare với ground truth
gt = frag_df.iloc[:test_psm.iloc[0]['nAA']-1]
print(f"\nGround truth shape: {gt.shape}")
print(f"Ground truth values: {gt.head()}")

TRAINING...
2026-02-27 14:42:57> Training with fixed sequence length: 0
[Training] Epoch=1, Mean Loss=0.014555285748263652
[Training] Epoch=2, Mean Loss=0.014553325039080599
[Training] Epoch=3, Mean Loss=0.014238429361615668
[Training] Epoch=4, Mean Loss=0.013904985971748829
[Training] Epoch=5, Mean Loss=0.013842066876928915

PREDICT 1 PSM...
Predicted shape: (11, 8)
Predicted values:         b_z1  b_z2      y_z1  y_z2  b_modloss_z1  b_modloss_z2  y_modloss_z1  \
0   0.000000   0.0  0.001474   0.0      0.000000           0.0      0.015210   
1   0.068167   0.0  0.077801   0.0      0.000000           0.0      0.187642   
2   0.014276   0.0  0.054420   0.0      0.000000           0.0      0.102145   
3   0.035178   0.0  0.133742   0.0      0.000000           0.0      0.271130   
4   0.052704   0.0  0.068210   0.0      0.000000           0.0      0.068269   
5   0.041920   0.0  0.076962   0.0      0.000000           0.0      0.062604   
6   0.013060   0.0  0.051863   0.0      0.000000    